# Yuk Ngintip Dapur Data: Praktik Simple ETL Pipeline

Selamat datang di sesi praktik **Data Engineering**!

Di notebook ini, kita bakal langsung praktik membangun **Data Pipeline sederhana** menggunakan pola klasik **ETL (Extract, Transform, Load)** dengan bahasa **Python** dan library **Pandas**.

### Skenario Kasus: Toko Online "Toko Gadget Nusantara"
Bayangin kamu baru masuk sebagai Junior Data Engineer di sebuah toko online. Setiap hari ada transaksi belanja yang tercatat di file mentah (`data/raw_transactions.csv`), sedangkan data profil pelanggan tersimpan terpisah di sistem lain (`data/customers.json`).

Tugas kita hari ini:
1. **Extract**: Mengambil data mentah dari dua sumber berbeda (CSV dan JSON), lalu mengecek kondisi kotornya.
2. **Transform**: Membersihkan, menyeragamkan format, menyaring, memperkaya (*enrich*), dan meringkas (*aggregate*) data di tahap *staging*.
3. **Load**: Memasukkan data yang sudah bersih dan siap pakai ke dalam simulasi *Data Warehouse* lokal (menggunakan SQLite) dan file CSV bersih.
4. **Preview Analyst**: Membuktikan bahwa setelah ETL selesai, Data Analyst tinggal menjalankan query SQL untuk menjawab pertanyaan bisnis tanpa pusing membersihkan data lagi.

In [1]:
# Download file yang dibutuhkan terlebih dahulu
!curl -L --create-dirs -o data/customers.json https://raw.githubusercontent.com/withered-flowers/works-demo-sjb/refs/heads/main/data/customers.json

!curl -L --create-dirs -o data/raw_transactions.csv https://raw.githubusercontent.com/withered-flowers/works-demo-sjb/refs/heads/main/data/raw_transactions.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1323  100  1323    0     0   8100      0 --:--:-- --:--:-- --:--:--  8166
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1611  100  1611    0     0  12425      0 --:--:-- --:--:-- --:--:-- 12488


## Persiapan Tools (Import Library)

Sebelum mulai "memasak" di dapur data, kita siapkan dulu peralatan yang dibutuhkan:
- `pandas`: Library paling populer di Python untuk mengolah data berbentuk tabel (mirip Excel, tapi jauh lebih cepat dan bisa diotomatisasi).
- `json`: Modul bawaan Python untuk membaca data berformat JSON.
- `sqlite3`: Modul bawaan Python untuk membuat dan mengakses database SQL ringan dalam bentuk satu file (kita pakai sebagai simulasi *Data Warehouse* lokal).
- `pathlib`: Modul bawaan Python untuk mengelola lokasi folder dan file dengan rapi.

In [2]:
import json
import sqlite3
from pathlib import Path
import pandas as pd

# Mengatur tampilan Pandas agar semua kolom terlihat rapi saat dicetak
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Persiapan selesai. Versi Pandas yang digunakan:", pd.__version__)

Persiapan selesai. Versi Pandas yang digunakan: 2.2.3


---
## Tahap 1: EXTRACT (Mengambil Bahan Mentah dari Hulu)

**Extract** adalah proses mengambil data dari berbagai sumber di hulu. Di dunia nyata, sumber data itu beragam: ada yang dari database transaksi (MySQL/PostgreSQL), API aplikasi, cloud storage, maupun file flat seperti CSV dan JSON.

Di sini kita punya dua sumber data:
1. `data/raw_transactions.csv` -> catatan transaksi harian toko online.
2. `data/customers.json` -> data master profil pelanggan.

In [ ]:
# 1. Extract data transaksi dari file CSV
raw_trx_path = DATA_DIR / "raw_transactions.csv"
df_raw = pd.read_csv(raw_trx_path)

print(f"Berhasil membaca {len(df_raw)} baris data transaksi mentah.")
df_raw.head(10)

Berhasil membaca 21 baris data transaksi mentah.


,transaction_id,transaction_date,customer_id,product_name,category,quantity,price_idr,status
0,TRX-001,2026-11-01,CUST-001,Mouse Wireless,aksesoris,2.0,150000,completed
1,TRX-002,01/11/2026,CUST-002,Keyboard Mechanical,Aksesoris,1.0,650000,COMPLETED
2,TRX-003,2026-11-01,CUST-003,laptop stand aluminum,Aksesoris,1.0,220000,Completed
3,TRX-003,2026-11-01,CUST-003,laptop stand aluminum,Aksesoris,1.0,220000,Completed
4,TRX-004,2026/11/02,CUST-001,Monitor 24 Inch,MONITOR,1.0,1850000,completed
5,TRX-005,02-11-2026,CUST-004,Webcam HD 1080p,aksesoris,2.0,350000,cancel
6,TRX-006,2026-11-02,CUST-005,USB Hub Type-C,Aksesoris,NaN,180000,completed
7,TRX-007,2026-11-03,CUST-002,Headset Gaming,Audio,1.0,480000,completed
8,TRX-008,03/11/2026,CUST-006,SSD Eksternal 1TB,Storage,1.0,1250000,Completed
9,TRX-009,2026-11-03,NaN,Mousepad XL,Aksesoris,3.0,85000,completed


In [ ]:
# 2. Extract data profil pelanggan dari file JSON
customers_path = DATA_DIR / "customers.json"
with open(customers_path, "r", encoding="utf-8") as file:
    customers_data = json.load(file)

df_customers = pd.DataFrame(customers_data)

print(f"Berhasil membaca {len(df_customers)} data profil pelanggan.")
df_customers

Berhasil membaca 8 data profil pelanggan.


,customer_id,customer_name,city,province,membership_tier
0,CUST-001,Andi Pratama,Jakarta Selatan,DKI Jakarta,Gold
1,CUST-002,Siti Rahma,Bandung,Jawa Barat,Platinum
2,CUST-003,Budi Santoso,Surabaya,Jawa Timur,Silver
3,CUST-004,Dina Kartika,Yogyakarta,DI Yogyakarta,Silver
4,CUST-005,Rizky Hidayat,Medan,Sumatera Utara,Gold
5,CUST-006,Nadia Putri,Semarang,Jawa Tengah,Platinum
6,CUST-007,Fajar Nugroho,Denpasar,Bali,Gold
7,CUST-008,Lestari Wulandari,Makassar,Sulawesi Selatan,Silver


### Inspeksi Awal Data Mentah (Kenali "Penyakit" Data)

Ingat prinsip penting di dunia data: **garbage in, garbage out**. Sebelum kita memproses data lebih jauh, seorang Data Engineer wajib mengecek dulu kondisi data mentahnya. Mari kita cek tipe data, nilai kosong (*missing values*), data duplikat, dan isi nilai uniknya.

In [ ]:
print("=== Informasi Struktur Data Mentah ===")
df_raw.info()

print("\n=== Jumlah Missing Values per Kolom ===")
print(df_raw.isna().sum())

print("\n=== Jumlah Baris Duplikat Persis ===")
print(df_raw.duplicated().sum())

print("\n=== Variasi Nilai pada Kolom 'status' ===")
print(df_raw["status"].unique())

=== Informasi Struktur Data Mentah ===
<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    21 non-null     str    
 1   transaction_date  21 non-null     str    
 2   customer_id       20 non-null     str    
 3   product_name      21 non-null     str    
 4   category          21 non-null     str    
 5   quantity          20 non-null     float64
 6   price_idr         21 non-null     int64  
 7   status            21 non-null     str    
dtypes: float64(1), int64(1), str(6)
memory usage: 1.4 KB

=== Jumlah Missing Values per Kolom ===
transaction_id      0
transaction_date    0
customer_id         1
product_name        0
category            0
quantity            1
price_idr           0
status              0
dtype: int64

=== Jumlah Baris Duplikat Persis ===
1

=== Variasi Nilai pada Kolom 'status' ===
<StringArray>
['completed', '

In [ ]:
# Mari tampilkan seluruh baris df_raw agar kita bisa melihat masalahnya dengan jelas
df_raw

,transaction_id,transaction_date,customer_id,product_name,category,quantity,price_idr,status
0,TRX-001,2026-11-01,CUST-001,Mouse Wireless,aksesoris,2.0,150000,completed
1,TRX-002,01/11/2026,CUST-002,Keyboard Mechanical,Aksesoris,1.0,650000,COMPLETED
2,TRX-003,2026-11-01,CUST-003,laptop stand aluminum,Aksesoris,1.0,220000,Completed
3,TRX-003,2026-11-01,CUST-003,laptop stand aluminum,Aksesoris,1.0,220000,Completed
4,TRX-004,2026/11/02,CUST-001,Monitor 24 Inch,MONITOR,1.0,1850000,completed
5,TRX-005,02-11-2026,CUST-004,Webcam HD 1080p,aksesoris,2.0,350000,cancel
6,TRX-006,2026-11-02,CUST-005,USB Hub Type-C,Aksesoris,NaN,180000,completed
7,TRX-007,2026-11-03,CUST-002,Headset Gaming,Audio,1.0,480000,completed
8,TRX-008,03/11/2026,CUST-006,SSD Eksternal 1TB,Storage,1.0,1250000,Completed
9,TRX-009,2026-11-03,NaN,Mousepad XL,Aksesoris,3.0,85000,completed


Dari inspeksi di atas, kita menemukan beberapa masalah klasik pada data mentah (persis seperti yang dibahas di materi *Data Preprocessing*):
1. **Data Duplikat**: `TRX-003` tercatat dua kali persis sama.
2. **Missing Values (Data Kosong)**: Baris `TRX-006` tidak punya `quantity`, dan `TRX-009` tidak punya `customer_id`.
3. **Nilai Ekstrem / Tidak Masuk Akal**: `TRX-010` punya `quantity = -1`, sedangkan `TRX-013` punya `price_idr = 0`.
4. **Format Tanggal Campur Aduk**: Ada yang `YYYY-MM-DD` (`2026-11-01`), `DD/MM/YYYY` (`01/11/2026`), `YYYY/MM/DD` (`2026/11/02`), dan `DD-MM-YYYY` (`02-11-2026`).
5. **Penulisan Teks Tidak Seragam**: Ada spasi berlebih (`"  Mouse Wireless  "`), serta huruf besar/kecil acak pada `status` (`"completed"`, `"COMPLETED"`, `" Completed "`) dan `category`.
6. **Status Transaksi Campuran**: Ada transaksi `cancel` dan `return` yang tidak boleh ikut dihitung sebagai penjualan sukses.

Semua masalah ini akan kita bereskan di tahap berikutnya: **Transform**!

---
## Tahap 2: TRANSFORM (Mengolah dan Membersihkan Bahan di Dapur)

Pada proses ETL klasik, tahap **Transform** dilakukan di area *staging* (di memori Python/Pandas) sebelum data masuk ke *Data Warehouse*.

Sesuai materi kita, ada 5 proses utama di tahap Transform:
1. **Cleaning**: Menghapus duplikat, menangani *missing values*, dan membuang nilai tidak masuk akal.
2. **Standardization**: Menyeragamkan format tanggal, kapitalisasi teks, spasi, dan tipe data.
3. **Filtering**: Membuang data yang tidak relevan (hanya ambil transaksi `completed`).
4. **Enrichment**: Menambahkan informasi pelengkap dari tabel pelanggan (`customers.json`) dan menghitung total belanja.
5. **Aggregation**: Membuat tabel ringkasan siap analisis (misalnya total penjualan per hari).

### 2.1 Cleaning (Membersihkan Data Kotor)

Kita mulai dengan menyalin `df_raw` ke variabel baru bernama `df_clean`. Tujuannya agar data asli tetap aman sebagai cadangan jika sewaktu-waktu kita perlu mengulang prosesnya.

In [ ]:
# Salin data mentah agar data sumber tidak berubah
df_clean = df_raw.copy()
awal = len(df_clean)

# 1. Hapus baris yang duplikat persis
df_clean = df_clean.drop_duplicates()
print(
    f"1. Setelah hapus duplikat: {len(df_clean)} baris (dibuang {awal - len(df_clean)})"
)

# 2. Tangani missing values pada kolom penting (customer_id, quantity, price_idr)
# Catatan: Untuk kasus transaksi keuangan/penjualan, baris tanpa ID pelanggan atau jumlah
# tidak bisa ditebak sembarangan dengan rata-rata, jadi paling aman kita buang (drop).
sebelum_na = len(df_clean)
df_clean = df_clean.dropna(subset=["customer_id", "quantity", "price_idr"])
print(
    f"2. Setelah hapus missing values: {len(df_clean)} baris (dibuang {sebelum_na - len(df_clean)})"
)

# 3. Buang nilai tidak masuk akal (outlier/error input: quantity <= 0 atau price_idr <= 0)
sebelum_valid = len(df_clean)
df_clean = df_clean[(df_clean["quantity"] > 0) & (df_clean["price_idr"] > 0)]
print(
    f"3. Setelah buang nilai <= 0: {len(df_clean)} baris (dibuang {sebelum_valid - len(df_clean)})"
)

1. Setelah hapus duplikat: 20 baris (dibuang 1)
2. Setelah hapus missing values: 18 baris (dibuang 2)
3. Setelah buang nilai <= 0: 16 baris (dibuang 2)


### 2.2 Standardization (Menyeragamkan Format)

Sekarang kita seragamkan teks dan format tanggalnya:
- Spasi di awal/akhir teks dibuang menggunakan `.str.strip()`.
- `product_name` dan `category` kita ubah jadi *Title Case* (huruf depan kapital).
- `status` kita ubah jadi huruf kecil semua (*lowercase*) agar `"COMPLETED"` dan `" Completed "` terbaca sama sebagai `"completed"`.
- Untuk tanggal (`transaction_date`), karena formatnya campur aduk antara `YYYY-MM-DD` dan `DD/MM/YYYY`, kita buat fungsi khusus `parse_mixed_date()`. Kenapa perlu fungsi khusus? Karena kalau langsung memakai `dayfirst=True` pada string `2026-11-01`, komputer bisa salah menebaknya menjadi 11 Januari, padahal maksudnya 1 November!

In [ ]:
def parse_mixed_date(date_text):
    """Mengubah beragam format tanggal menjadi format standar datetime."""
    text = str(date_text).strip()
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(text, format=fmt)
        except ValueError:
            continue
    return pd.NaT


# Standardisasi teks
df_clean["product_name"] = df_clean["product_name"].str.strip().str.title()
df_clean["category"] = df_clean["category"].str.strip().str.title()
df_clean["status"] = df_clean["status"].str.strip().str.lower()
df_clean["customer_id"] = df_clean["customer_id"].str.strip().str.upper()

# Standardisasi tanggal menjadi string format YYYY-MM-DD
df_clean["transaction_date"] = (
    df_clean["transaction_date"].apply(parse_mixed_date).dt.strftime("%Y-%m-%d")
)

# Pastikan kolom angka bertipe integer
df_clean["quantity"] = df_clean["quantity"].astype(int)
df_clean["price_idr"] = df_clean["price_idr"].astype(int)

print("Variasi status setelah standardisasi:", df_clean["status"].unique())
df_clean[
    ["transaction_id", "transaction_date", "product_name", "category", "status"]
].head(10)

Variasi status setelah standardisasi: <StringArray>
['completed', 'cancel', 'return']
Length: 3, dtype: str


,transaction_id,transaction_date,product_name,category,status
0,TRX-001,2026-11-01,Mouse Wireless,Aksesoris,completed
1,TRX-002,2026-11-01,Keyboard Mechanical,Aksesoris,completed
2,TRX-003,2026-11-01,Laptop Stand Aluminum,Aksesoris,completed
4,TRX-004,2026-11-02,Monitor 24 Inch,Monitor,completed
5,TRX-005,2026-11-02,Webcam Hd 1080P,Aksesoris,cancel
7,TRX-007,2026-11-03,Headset Gaming,Audio,completed
8,TRX-008,2026-11-03,Ssd Eksternal 1Tb,Storage,completed
11,TRX-011,2026-11-04,Kabel Hdmi 2M,Aksesoris,return
12,TRX-012,2026-11-04,Keyboard Mechanical,Aksesoris,completed
14,TRX-014,2026-11-05,Usb Hub Type-C,Aksesoris,completed


### 2.3 Filtering (Menyaring Data yang Relevan)

Untuk kebutuhan laporan omzet penjualan bersih, transaksi dengan status `cancel` atau `return` tidak boleh ikut dihitung. Jadi kita saring dan ambil yang berstatus `"completed"` saja.

In [ ]:
sebelum_filter = len(df_clean)
df_clean = df_clean[df_clean["status"] == "completed"].copy()

print(
    f"Setelah filter status == 'completed': {len(df_clean)} baris (dibuang {sebelum_filter - len(df_clean)} baris cancel/return)"
)
df_clean.reset_index(drop=True, inplace=True)
df_clean

Setelah filter status == 'completed': 13 baris (dibuang 3 baris cancel/return)


,transaction_id,transaction_date,customer_id,product_name,category,quantity,price_idr,status
0,TRX-001,2026-11-01,CUST-001,Mouse Wireless,Aksesoris,2,150000,completed
1,TRX-002,2026-11-01,CUST-002,Keyboard Mechanical,Aksesoris,1,650000,completed
2,TRX-003,2026-11-01,CUST-003,Laptop Stand Aluminum,Aksesoris,1,220000,completed
3,TRX-004,2026-11-02,CUST-001,Monitor 24 Inch,Monitor,1,1850000,completed
4,TRX-007,2026-11-03,CUST-002,Headset Gaming,Audio,1,480000,completed
5,TRX-008,2026-11-03,CUST-006,Ssd Eksternal 1Tb,Storage,1,1250000,completed
6,TRX-012,2026-11-04,CUST-008,Keyboard Mechanical,Aksesoris,2,650000,completed
7,TRX-014,2026-11-05,CUST-001,Usb Hub Type-C,Aksesoris,3,180000,completed
8,TRX-015,2026-11-05,CUST-005,Mouse Wireless,Aksesoris,2,150000,completed
9,TRX-016,2026-11-06,CUST-006,Webcam Hd 1080P,Aksesoris,1,350000,completed


### 2.4 Enrichment (Memperkaya Data dengan Informasi Tambahan)

Di tabel transaksi, kita baru punya `customer_id` (misalnya `CUST-001`), belum tahu siapa nama pelanggannya, dari kota mana, dan apa level membership-nya.

Di sinilah proses **Enrichment** berperan: kita gabungkan (`merge`) data transaksi dengan `df_customers` berdasarkan `customer_id`, lalu kita tambahkan kolom turunan `total_amount` (`quantity * price_idr`).

In [ ]:
# Gabungkan data transaksi bersih dengan profil pelanggan
df_enriched = pd.merge(df_clean, df_customers, on="customer_id", how="left")

# Hitung total nominal per transaksi
df_enriched["total_amount"] = df_enriched["quantity"] * df_enriched["price_idr"]

# Rapikan urutan kolom agar mudah dibaca
kolom_rapi = [
    "transaction_id",
    "transaction_date",
    "customer_id",
    "customer_name",
    "city",
    "province",
    "membership_tier",
    "product_name",
    "category",
    "quantity",
    "price_idr",
    "total_amount",
    "status",
]
df_enriched = (
    df_enriched[kolom_rapi]
    .sort_values(by=["transaction_date", "transaction_id"])
    .reset_index(drop=True)
)

print(
    f"Data setelah enrichment memiliki {df_enriched.shape[0]} baris dan {df_enriched.shape[1]} kolom."
)
df_enriched

Data setelah enrichment memiliki 13 baris dan 13 kolom.


,transaction_id,transaction_date,customer_id,customer_name,city,province,membership_tier,product_name,category,quantity,price_idr,total_amount,status
0,TRX-001,2026-11-01,CUST-001,Andi Pratama,Jakarta Selatan,DKI Jakarta,Gold,Mouse Wireless,Aksesoris,2,150000,300000,completed
1,TRX-002,2026-11-01,CUST-002,Siti Rahma,Bandung,Jawa Barat,Platinum,Keyboard Mechanical,Aksesoris,1,650000,650000,completed
2,TRX-003,2026-11-01,CUST-003,Budi Santoso,Surabaya,Jawa Timur,Silver,Laptop Stand Aluminum,Aksesoris,1,220000,220000,completed
3,TRX-004,2026-11-02,CUST-001,Andi Pratama,Jakarta Selatan,DKI Jakarta,Gold,Monitor 24 Inch,Monitor,1,1850000,1850000,completed
4,TRX-007,2026-11-03,CUST-002,Siti Rahma,Bandung,Jawa Barat,Platinum,Headset Gaming,Audio,1,480000,480000,completed
5,TRX-008,2026-11-03,CUST-006,Nadia Putri,Semarang,Jawa Tengah,Platinum,Ssd Eksternal 1Tb,Storage,1,1250000,1250000,completed
6,TRX-012,2026-11-04,CUST-008,Lestari Wulandari,Makassar,Sulawesi Selatan,Silver,Keyboard Mechanical,Aksesoris,2,650000,1300000,completed
7,TRX-014,2026-11-05,CUST-001,Andi Pratama,Jakarta Selatan,DKI Jakarta,Gold,Usb Hub Type-C,Aksesoris,3,180000,540000,completed
8,TRX-015,2026-11-05,CUST-005,Rizky Hidayat,Medan,Sumatera Utara,Gold,Mouse Wireless,Aksesoris,2,150000,300000,completed
9,TRX-016,2026-11-06,CUST-006,Nadia Putri,Semarang,Jawa Tengah,Platinum,Webcam Hd 1080P,Aksesoris,1,350000,350000,completed


### 2.5 Aggregation (Membuat Tabel Ringkasan Siap Analisis)

Salah satu tugas penting Data Engineer (dan *Analytics Engineer*) adalah menyiapkan tabel ringkasan (*summary table* / *data mart*) agar tim Data Analyst atau dashboard BI bisa langsung membacanya dengan cepat.

Mari kita buat dua tabel agregasi:
1. `df_daily_sales`: Ringkasan jumlah transaksi, total barang terjual, dan total pendapatan per hari.
2. `df_city_sales`: Ringkasan penjualan berdasarkan kota pelanggan.

In [7]:
# Agregasi 1: Total penjualan per hari
df_daily_sales = (
    df_enriched.groupby("transaction_date", as_index=False)
    .agg(
        total_transactions=("transaction_id", "count"),
        total_items_sold=("quantity", "sum"),
        total_revenue_idr=("total_amount", "sum"),
    )
    .sort_values("transaction_date")
)

print("=== Tabel Ringkasan Penjualan Harian ===")
df_daily_sales

NameError: name 'df_enriched' is not defined

In [6]:
# Agregasi 2: Total penjualan per kota dan provinsi
df_city_sales = (
    df_enriched.groupby(["city", "province"], as_index=False)
    .agg(
        total_transactions=("transaction_id", "count"),
        total_revenue_idr=("total_amount", "sum"),
    )
    .sort_values("total_revenue_idr", ascending=False)
    .reset_index(drop=True)
)

print("=== Tabel Ringkasan Penjualan per Kota ===")
df_city_sales

NameError: name 'df_enriched' is not defined

---
## Tahap 3: LOAD (Menyajikan Data Matang ke Hilir)

Setelah bahan mentah selesai dimasak dan dirapikan, sekarang saatnya tahap **Load**: memasukkan data matang ke tempat tujuan akhir (*Data Warehouse* atau *Data Mart*).

Dalam praktiknya, ada tiga strategi Load yang sering dipakai:
- **Full Load**: Menghapus data lama di tabel tujuan dan menggantinya dengan seluruh data baru yang bersih. Simpel, cocok untuk data berukuran kecil-menengah.
- **Incremental Load**: Hanya memasukkan data yang baru atau berubah sejak proses sebelumnya. Jauh lebih hemat resource untuk data besar.
- **Upsert (Update + Insert)**: Jika datanya sudah ada maka diperbarui, jika belum ada maka ditambahkan.

Hal terpenting saat Load adalah memastikan prosesnya bersifat **atomic**: kalau terjadi error di tengah jalan, perubahan dibatalkan (*rollback*) sehingga tidak ada data setengah jadi yang masuk ke database.

Di notebook ini, kita menggunakan strategi **Full Load** ke dalam database SQLite (`output/data_warehouse.db`) dengan blok transaksi `with conn:` agar prosesnya tetap *atomic*, sekaligus mengekspor file CSV bersih ke `output/clean_transactions.csv`.

In [5]:
db_path = OUTPUT_DIR / "data_warehouse.db"
csv_output_path = OUTPUT_DIR / "clean_transactions.csv"

# 1. Load ke simulasi Data Warehouse (SQLite) secara atomic
conn = sqlite3.connect(db_path)
try:
    with conn:
        df_enriched.to_sql(
            "fact_transactions_clean", conn, if_exists="replace", index=False
        )
        df_daily_sales.to_sql(
            "mart_daily_sales", conn, if_exists="replace", index=False
        )
        df_city_sales.to_sql("mart_city_sales", conn, if_exists="replace", index=False)
    print(f"Berhasil memuat 3 tabel ke Data Warehouse: {db_path}")
finally:
    conn.close()

# 2. Simpan juga versi flat-file CSV bersih sebagai backup/kebutuhan ad-hoc
df_enriched.to_csv(csv_output_path, index=False)
print(f"Berhasil menyimpan file CSV bersih ke: {csv_output_path}")

NameError: name 'df_enriched' is not defined

---
## Tahap 4: Simulasi Kerja Data Analyst (Descriptive Analytics)

Masih ingat materi kita tentang pembagian peran di dapur data?
- **Data Engineer** menyiapkan *infrastructure* dan *pipeline* (yang baru saja kita selesaikan!).
- **Data Analyst** bertugas melakukan *reporting* dan menjawab pertanyaan bisnis (*Descriptive Analytics*: *"Apa yang terjadi?"*) dengan cara melakukan query ke *Data Warehouse* yang sudah bersih.

Mari kita buktikan bahwa sekarang Data Analyst tinggal menulis query SQL sederhana ke `output/data_warehouse.db` tanpa perlu lagi pusing mengurus format tanggal yang berantakan atau baris duplikat.

In [8]:
from IPython.display import display

conn = sqlite3.connect(db_path)

# Pertanyaan Bisnis 1: Produk apa yang menghasilkan pendapatan terbesar?
query_top_products = """
SELECT
    product_name,
    category,
    SUM(quantity) AS total_unit_terjual,
    SUM(total_amount) AS total_pendapatan_idr
FROM fact_transactions_clean
GROUP BY product_name, category
ORDER BY total_pendapatan_idr DESC;
"""

df_top_products = pd.read_sql_query(query_top_products, conn)
print("=== Analisis 1: Performa Penjualan per Produk ===")
display(df_top_products)

# Pertanyaan Bisnis 2: Bagaimana kontribusi omzet berdasarkan membership tier?
query_tier = """
SELECT
    membership_tier,
    COUNT(transaction_id) AS jumlah_transaksi,
    SUM(total_amount) AS total_pendapatan_idr,
    ROUND(AVG(total_amount), 0) AS rata_rata_belanja_idr
FROM fact_transactions_clean
GROUP BY membership_tier
ORDER BY total_pendapatan_idr DESC;
"""

df_tier_summary = pd.read_sql_query(query_tier, conn)
conn.close()

print("\n=== Analisis 2: Performa Penjualan per Membership Tier ===")
display(df_tier_summary)

DatabaseError: Execution failed on sql '
SELECT
    product_name,
    category,
    SUM(quantity) AS total_unit_terjual,
    SUM(total_amount) AS total_pendapatan_idr
FROM fact_transactions_clean
GROUP BY product_name, category
ORDER BY total_pendapatan_idr DESC;
': no such table: fact_transactions_clean

---
## Penutup dan Tantangan Latihan

Selamat! Kamu baru saja membangun sebuah **Simple ETL Pipeline** yang utuh dari hulu ke hilir:
1. **Extract**: Membaca data transaksi kotor (`raw_transactions.csv`) dan data profil pelanggan (`customers.json`).
2. **Transform**: Membersihkan duplikat, *missing values*, dan outlier; menyeragamkan teks dan format tanggal; menyaring transaksi `completed`; memperkaya data dengan *join* dan kalkulasi `total_amount`; serta membuat tabel agregasi.
3. **Load**: Menyimpan hasilnya secara *atomic* ke database SQLite (`output/data_warehouse.db`) dan file CSV bersih (`output/clean_transactions.csv`).
4. **Query Analyst**: Menguji hasil *Data Warehouse* menggunakan SQL untuk menghasilkan informasi bisnis.

### Tantangan Latihan Mandiri (Opsional)
Kalau kamu mau mengasah pemahamanmu lebih jauh, coba kerjakan tantangan kecil berikut di cell baru di bawah:
1. **Modularisasi Pipeline**: Bungkus langkah-langkah di atas menjadi tiga fungsi Python terpisah: `extract_data()`, `transform_data(df_raw, df_customers)`, dan `load_data(df_enriched, db_path)` supaya kodenya siap dijadwalkan (misalnya nanti di tools orchestrator seperti Apache Airflow).
2. **Tambah Tabel Agregasi Baru**: Buatlah tabel `mart_category_sales` yang merangkum total pendapatan per `category`, lalu simpan ke `output/data_warehouse.db`.
3. **Eksplorasi SQL**: Coba tulis query SQL untuk mencari transaksi dengan nilai `total_amount` paling besar dan siapa nama pelanggannya.